# 02. Parametric Surge Model Calibration against IBTrACS & Observed Landfalls

This notebook calibrates the inverted barometer and bathymetric amplification factors of the parametric surge model against observed peak storm surges from:
- **Cyclone Fani (2019)**: ~1.5m - 2.0m surge in Puri
- **Cyclone Amphan (2020)**: ~4.5m - 5.0m surge in Sundarbans / Digha
- **Cyclone Mocha (2023)**: ~3.0m - 3.5m surge in Sittwe / Cox's Bazar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from modeling.surge.parametric_surge import calculate_surge_height, shelf_amplification

# Calibration dataset: (Name, Delta P (hPa), RMW (km), Shelf Slope, Observed Surge (m))
historical_events = [
    ("Fani (2019)", 81.0, 35.0, 0.0015, 1.8),
    ("Amphan (2020)", 90.0, 45.0, 0.0008, 4.6),
    ("Mocha (2023)", 85.0, 30.0, 0.0012, 3.2),
]

print("Event | Modeled Surge (m) | Observed Surge (m) | Residual (m)")
print("-" * 55)
for name, dp, rmw, slope, obs in historical_events:
    modeled = calculate_surge_height(dp, rmw, 130.0) * shelf_amplification(slope)
    diff = modeled - obs
    print(f"{name:14} | {modeled:17.2f} | {obs:18.2f} | {diff:10.2f}")

## Visualizing Pressure Deficit vs Modeled Peak Inundation

In [ ]:
pressures = np.linspace(20, 100, 50)
surges_flat = [calculate_surge_height(p, 40.0, 120.0) * shelf_amplification(0.0008) for p in pressures]
surges_steep = [calculate_surge_height(p, 40.0, 120.0) * shelf_amplification(0.0030) for p in pressures]

plt.figure(figsize=(8, 4))
plt.plot(pressures, surges_flat, label="Gentle Shelf (Bengal Delta)", color="#ef4444", lw=2)
plt.plot(pressures, surges_steep, label="Steep Shelf (Peninsular East Coast)", color="#38bdf8", lw=2)
plt.xlabel("Central Pressure Deficit ΔP (hPa)")
plt.ylabel("Peak Storm Surge (m)")
plt.title("Parametric Surge vs Pressure Deficit by Coastal Slope")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()